### Ingestión del archivo "language_role.json"
#### Este notebook es un clon del 08-Ingestion File language_role de la carpeta ingestion con algunas cosas modificadas para que sea carga incremental, las cosas modificadas tienen el comentario al lado, básicamente lo que se modifica es:
1. Se agrega el widget v_file_date con el nombre de las carpetas de carga incremental del contenedor bronze-inc
2. Se modifica la creación del language_role_df, usando la variable del contenedor de bronze incremental y apuntando al widget v_file_date
3. Se agrega la columna file_date
4. Se elimina el paso "Escribir datos en el datalake en formato Parquet"
5. Se cambia el esquema movie_silver por movie_silver_inc a la hora de crear la managed table

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

###Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark
####La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

language_role_schema = StructType(fields = [
    StructField("roleId", IntegerType(), True),
    StructField("languageRole", StringType(), True)
])


In [0]:
language_role_df = spark.read.schema(language_role_schema).option("multiLine", True).json(f"{bronze_inc_folder_path}/{v_file_date}/language_role.json")


In [0]:
language_role_df.printSchema()

In [0]:
display(language_role_df)

### Paso 2 - Renombrar las columnas y añadir columnas "ingestion_date", "environment" y "file_date"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

language_role_final_df = language_role_df \
                             .withColumnsRenamed({"roleId": "role_id",
                                             "languageRole": "language_role"})

language_role_final_df = add_ingestion_date(language_role_final_df) \
                             .withColumn("environment", lit(v_environment)) \
                             .withColumn("file_date", lit(v_file_date))  

In [0]:
display(language_role_final_df)

### Paso 3 - Escribir datos en una managed table en el contenedor silver


In [0]:
language_role_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver_inc.languages_roles")

In [0]:
%sql
SELECT * FROM movie_silver_inc.languages_roles

In [0]:
dbutils.notebook.exit("Success")